# 🗣️ 05 - Choosing and Telling

You can now draw almost any chart. This notebook is about drawing the **right** one, keeping it **honest**, and making it say **one clear thing**.

The difference between an exploration chart and a communication chart:

| | Exploration (for you) | Communication (for someone else) |
|---|---|---|
| Goal | find out what's in the data | show one finding clearly |
| How many | dozens, fast | one or two, polished |
| Title | describes the chart ("Fare by Class") | states the finding ("1st class paid 7× more") |
| Example | `pairplot`, quick `hue` histograms | the annotated, labeled chart in a report |

Notebooks 01–04 were mostly exploration. This one is communication.

## ✅ Learning Goals
- Choose a chart from the **question** and the **data types** (Module 07 notebook 01)
- Keep scales honest: zero-based bars, shared axes, and the same color scale across charts
- Order categories on purpose, and use color to **highlight** rather than decorate
- Use colorblind-safe palettes and consistent colors for the same category
- Write titles that state the finding, and label data directly instead of relying on legends
- Turn a default chart into a clear one, step by step

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from pathlib import Path

OUT = Path('05-OutputFiles')
OUT.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
penguins = pd.read_csv('penguins.csv').dropna(subset=MEASUREMENTS)
titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})
poke = pd.read_csv('pokemon.csv')

## 1. Start from the Question
Choose the chart from **what you're asking** and **what kind of columns** you have (Module 07 notebook 01):

| Question | Column types | Chart | Seaborn / pandas |
|---|---|---|---|
| How is one numeric column distributed? | 1 numeric | histogram, KDE, ECDF | `histplot`, `kdeplot`, `ecdfplot` |
| How does that distribution differ by group? | numeric + categorical | box, violin, strip, or small multiples | `boxplot`, `violinplot`, `displot(col=)` |
| How many in each category? | 1 categorical | bar (sorted, or in ordinal order) | `countplot`, `value_counts().plot.barh()` |
| What's the average/rate per group? | numeric or 0/1 + categorical | bar or point, with error bars | `barplot`, `pointplot` |
| What's the mix within each group? | 2 categorical | 100% stacked bar | `crosstab(normalize='index').plot.barh(stacked=True)` |
| Are two numeric columns related? | 2 numeric | scatter (+ trend line, colored by group) | `scatterplot`, `regplot`, `lmplot` |
| How do many columns relate at once? | many numeric | heatmap, pair plot | `heatmap`, `pairplot` |
| How does something change over an ordered x? | numeric + ordered | line | `.plot()`, `lineplot` |

> 🧠 If you can't say the question in one sentence, you're not ready to choose the chart.

## 2. Honest Scales: Bars Start at Zero
A bar shows a value by its **length**. Cut off the bottom of the axis and the lengths lie.

In [ ]:
mass = penguins.groupby('species')['body_mass_g'].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')

mass.plot.bar(ax=axes[0], rot=0, color='tab:red')
axes[0].set(ylim=(3600, 5200), title='MISLEADING: axis starts at 3,600 g', xlabel='', ylabel='Mean body mass (g)')

mass.plot.bar(ax=axes[1], rot=0, color='tab:blue')
axes[1].set(ylim=(0, 5500), title='HONEST: axis starts at 0', xlabel='', ylabel='Mean body mass (g)')

plt.show()
print(mass.round(0))

On the left, the Chinstrap bar looks about **30% taller** than the Adelie bar. The real difference is about 30 g out of 3,700, under 1%. On the right, the bars look as similar as they really are, and Gentoo's real lead is still obvious.

The rules:
- **Bars and areas** encode length, so they must start at **zero**.
- **Lines, points, and box plots** encode **position**, so a non-zero axis is fine when it helps show the variation, as long as the axis is labeled.
- **Panels being compared** must share the axis (`sharey=True`, notebook 01), and **heatmaps** being compared must share `vmin`/`vmax` (notebook 04).

## 3. Order on Purpose

In [ ]:
med = poke.groupby('Type 1')['Total'].median()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), layout='constrained')

med.plot.barh(ax=axes[0], color='gray')
axes[0].set(title='Alphabetical: which type is highest?', xlabel='Median Total', ylabel='')

med.sort_values().plot.barh(ax=axes[1], color='tab:blue')
axes[1].set(title='Sorted: the ranking is instant', xlabel='Median Total', ylabel='')

plt.show()

| Category type | Order by |
|---|---|
| **Nominal** (types, species, ports) | value, largest first (or top-down for `barh`) |
| **Ordinal** (class, size, rating) | its natural order, **never** by value |
| Time, generations | chronological |

Pass `order=` to Seaborn functions to set the order explicitly (notebook 03).

## 4. Color: Highlight, Don't Decorate
Every color should **mean** something. If the bars are all the same kind of thing, they should all be the same color, except the one you want the reader to look at:

In [ ]:
sorted_med = med.sort_values()
colors = ['tab:orange' if t == 'Dragon' else 'lightgray' for t in sorted_med.index]

fig, ax = plt.subplots(figsize=(7, 5.5))
sorted_med.plot.barh(ax=ax, color=colors)
ax.set(title='Dragon Types Have the Highest Median Total', xlabel='Median Total', ylabel='')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

The reader's eye goes straight to the orange bar, and the title tells them why.

### Colorblind-safe palettes
About 1 in 12 men has some form of color vision deficiency, most often red-green. Seaborn's `'colorblind'` palette keeps categories distinguishable:

In [ ]:
sns.palplot(sns.color_palette('colorblind'))
plt.title("sns.color_palette('colorblind')")
plt.show()

### The same category, the same color, everywhere
If Gentoo is green in one chart, it must be green in every chart of the report. Build a **dictionary palette** once and pass it to every chart with `palette=`:

In [ ]:
SPECIES_COLORS = dict(zip(['Adelie', 'Chinstrap', 'Gentoo'], sns.color_palette('colorblind', 3)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
sns.histplot(data=penguins, x='flipper_length_mm', hue='species', palette=SPECIES_COLORS,
             element='step', ax=axes[0])
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', hue='species',
                palette=SPECIES_COLORS, ax=axes[1], legend=False)
axes[0].set(title='Flipper length')
axes[1].set(title='Bill shape')
plt.show()

## 5. Titles That Say Something, and Labels Where the Data Are
A **descriptive** title ("Survival Rate by Class") makes the reader work out the point. A **finding** title ("Women in 1st and 2nd Class Almost All Survived") tells them, and the chart is the evidence.

**Direct labels** (the value on the bar, the name at the end of the line) beat a legend the reader has to look back and forth at:

In [ ]:
rates = titanic.pivot_table(values='Survived', index='class', columns='Sex', aggfunc='mean')
colors = {'female': 'tab:orange', 'male': 'tab:blue'}

fig, ax = plt.subplots(figsize=(7, 4.5))
for sex in ['female', 'male']:
    ax.plot(rates.index, rates[sex], marker='o', linewidth=2.5, color=colors[sex])
    ax.text(2.08, rates[sex].iloc[-1], sex, color=colors[sex], va='center', fontweight='bold')

ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(ylim=(0, 1.05), xlim=(-0.2, 2.4), xlabel='Ticket class', ylabel='Survived')
ax.set_title('Women in 1st and 2nd Class Almost All Survived', loc='left', fontweight='bold', pad=24)
ax.text(0, 1.03, 'Titanic survival rate by ticket class and sex', transform=ax.transAxes, color='gray')
ax.spines[['top', 'right']].set_visible(False)
plt.show()

- `ax.text(x, y, ...)` places the labels just past the last point, in data coordinates.
- `transform=ax.transAxes` switches to **axes** coordinates, where (0, 0) is the bottom-left corner of the plot and (1, 1) the top-right. That's handy for a subtitle that stays in place, whatever the data.
- A bold, left-aligned finding as the title (`pad=24` leaves room under it), with a gray subtitle for the details, is a common layout in professional reports.

## 6. A Makeover, Step by Step
Here's a quick, default chart of a real question: **"Who survived the Titanic?"**

In [ ]:
ax = titanic.groupby(['class', 'Sex'])['Survived'].mean().plot.bar()
plt.savefig(OUT / 'makeover_before.png', dpi=120, bbox_inches='tight')
plt.show()

What's wrong with it:
1. The title is missing, so there's no question and no finding.
2. The y-axis is unlabeled, and shows 0.0–1.0 instead of percentages.
3. The tick labels are rotated tuples, like `(1st, female)`.
4. Every bar is the same color, so the sexes don't stand out.
5. You can't read exact values.

The makeover fixes each one:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=titanic, x='class', y='Survived', hue='Sex', order=['1st', '2nd', '3rd'],
            palette=colors, errorbar=None, ax=ax)

for bars in ax.containers:
    ax.bar_label(bars, labels=[f'{v:.0%}' for v in bars.datavalues], padding=3, fontsize=10)

ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(ylim=(0, 1.1), xlabel='', ylabel='Survived')
ax.set_title('On the Titanic, Sex Mattered More Than Class', loc='left', fontweight='bold', pad=24)
ax.text(0, 1.03, 'Survival rate by ticket class and sex (891 passengers)',
        transform=ax.transAxes, color='gray')
ax.legend(title='', frameon=False, loc='upper right')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

fig.savefig(OUT / 'makeover_after.png', dpi=150, bbox_inches='tight')
plt.show()

Same data, same bars. Now the chart answers the question before the reader even looks at the bars. `errorbar=None` was a choice: with direct labels and a general audience, the error bars added clutter. For a technical audience you'd keep them and say so (notebook 04).

## 7. The Final Checklist
Before you share a chart (this extends the Foundations 07 checklist):

- [ ] **Question**: can I say in one sentence what this chart answers?
- [ ] **Type**: does the chart type fit the question and the column types (section 1)?
- [ ] **Title**: does it state the finding, not just describe the chart?
- [ ] **Labels**: are the axes labeled, with units? Is the source or sample size given?
- [ ] **Scale**: do bars start at zero? Do compared panels and heatmaps share their scales?
- [ ] **Order**: are nominal categories sorted by value, and ordinal ones in their natural order?
- [ ] **Color**: does every color mean something? Is it colorblind-safe, and consistent across charts?
- [ ] **Clutter**: can any legend, gridline, border, or decoration be removed? Can data be labeled directly?
- [ ] **Honesty**: does the chart show the missing values, small groups, or outliers it might hide?

---
### 📝 Summary
- Pick the chart from the **question** and the **column types**.
- Bars start at zero. Compared panels share axes, and compared heatmaps share color scales.
- Sort nominal categories by value, and keep ordinal ones in order.
- Use color to **highlight**, keep it colorblind-safe, and keep the same color for the same category.
- Titles state the finding. Label data directly when you can.
- Exploration charts are for you. Communication charts get the full checklist.

## 🏋️ Practice

### Practice 1 — Fix the Axis (easy)
Plot the **median** `Fare` per `Embarked` port as a bar chart with the y-axis set to `ylim=(7, 30)`. Then fix it. In a sentence, say how the misleading version would mislead a reader.

In [ ]:
# Your code here

*Your sentence here.*

### Practice 2 — Highlight One Group (medium)
Make a sorted horizontal bar chart of the **Legendary rate** per `Type 1` (Pokémon). Highlight the type with the highest rate in a strong color and gray out the rest. Write a title that states the finding.

In [ ]:
# Your code here

### Practice 3 — Pick the Chart (harder)
For each question below, name the chart type and the Seaborn/pandas function, **then make it**:
1. Do Legendary Pokémon have higher `Speed` than non-Legendary ones, and how much do they overlap?
2. How has the share of each `Type 1` changed across `Generation`s, for the 4 most common types?
3. Is `HP` related to `Defense`, and is the relationship different for Legendary Pokémon?

In [ ]:
# Your code here

## ✏️ Your Turn
Do your own **makeover**. Start with this default chart:
```python
penguins.groupby(['island', 'species'])['body_mass_g'].mean().plot.bar()
```
Decide what question it should answer (hint: Module 07's lab showed island and species are tangled together), then redesign it using the checklist. Save **both** versions as `OUT / 'my_makeover_before.png'` and `OUT / 'my_makeover_after.png'`, and list the checklist items you fixed.

In [ ]:
# Your turn here

*Checklist items you fixed:*

## 🔥 Challenge (Optional) — A Chart That Lies, on Purpose
Using any dataset from this module, make a chart that is technically accurate but **misleading** (truncated axis, cherry-picked groups, a misleading color scale, a dual axis, or a mean that hides a bimodal distribution). Then make the honest version next to it. In a Markdown cell, explain the trick and how a careful reader could have caught it.

In [ ]:
# Your code here

*Your explanation here.*

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `05-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
✅ Next up → [Visualization Lab](06-Visualization_Lab.ipynb): back to the penguins, to turn your Module 07 report into a set of figures that tell its story.